[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_1_2/06_tag_1_2_unueberwachtes_lernen_clustering.ipynb)

# Tag 1-2 - 06 Unüberwachtes Lernen und Clustering

Unüberwachtes Lernen bekommt nur `X`. Die Anzahl Cluster ist ein Hyperparameter und wird hier wieder per Regler getestet.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
print('Setup abgeschlossen.')


In [ ]:
def make_unlabeled_machine_data(cluster_overlap=0.20, seed=RANDOM_STATE):
    local_rng = np.random.default_rng(seed)
    centers = np.array([[0.2, 0.3], [1.6, 0.5], [0.9, 1.7]])
    scale = 0.14 + cluster_overlap
    parts = [local_rng.normal(center, [scale, scale], size=(70, 2)) for center in centers]
    X = np.vstack(parts)
    order = local_rng.permutation(len(X))
    return pd.DataFrame(X[order], columns=['Schwingung', 'Temperaturtrend'])

def standardize(X):
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    return (X - mean) / std, mean, std

def kmeans(X, k=3, steps=20, seed=RANDOM_STATE):
    local_rng = np.random.default_rng(seed)
    centers = X[local_rng.choice(len(X), size=k, replace=False)].copy()
    labels = np.zeros(len(X), dtype=int)
    for _ in range(steps):
        distances = np.sqrt(((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2))
        labels = distances.argmin(axis=1)
        for cluster_id in range(k):
            mask = labels == cluster_id
            if np.any(mask):
                centers[cluster_id] = X[mask].mean(axis=0)
    return labels, centers

def inertia_score(X, labels, centers):
    return float(np.sum((X - centers[labels]) ** 2))

def silhouette_score_simple(X, labels):
    unique_labels = np.unique(labels)
    if len(unique_labels) < 2 or len(unique_labels) >= len(X):
        return np.nan
    distances = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))
    values = []
    for i in range(len(X)):
        same = labels == labels[i]
        same[i] = False
        a = distances[i, same].mean() if np.any(same) else 0.0
        b = min(distances[i, labels == other].mean() for other in unique_labels if other != labels[i])
        values.append((b - a) / max(a, b) if max(a, b) > 0 else 0.0)
    return float(np.mean(values))


In [ ]:
def clustering_demo(clusteranzahl=3, cluster_overlap=0.20, seed=42):
    df = make_unlabeled_machine_data(cluster_overlap=cluster_overlap, seed=seed)
    X_raw = df[['Schwingung', 'Temperaturtrend']].to_numpy()
    X_scaled, X_mean, X_std = standardize(X_raw)

    metric_rows = []
    results = {}
    for k in range(2, 9):
        labels_k, centers_k = kmeans(X_scaled, k=k, seed=seed)
        inertia = inertia_score(X_scaled, labels_k, centers_k)
        silhouette = silhouette_score_simple(X_scaled, labels_k)
        metric_rows.append({'k': k, 'Inertia': inertia, 'Silhouette': silhouette})
        results[k] = (labels_k, centers_k)
    metrics_df = pd.DataFrame(metric_rows)
    best_k = int(metrics_df.loc[metrics_df['Silhouette'].idxmax(), 'k'])

    labels, centers_scaled = results[clusteranzahl]
    centers_raw = centers_scaled * X_std + X_mean
    current = metrics_df[metrics_df['k'] == clusteranzahl].iloc[0]
    display(metrics_df.round(3))
    print(f'Bestes k laut Silhouette: {best_k} | aktuell k={clusteranzahl}, Silhouette={current.Silhouette:.3f}, Inertia={current.Inertia:.1f}')

    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    axes[0].scatter(df['Schwingung'], df['Temperaturtrend'], c=labels, cmap='tab10', alpha=0.75)
    axes[0].scatter(centers_raw[:, 0], centers_raw[:, 1], marker='X', s=220, color='black', label='Clusterzentren')
    axes[0].set_xlabel('Schwingung')
    axes[0].set_ylabel('Temperaturtrend')
    axes[0].set_title(f'K-Means mit k={clusteranzahl}')
    axes[0].legend()

    axes[1].plot(metrics_df['k'], metrics_df['Inertia'], marker='o')
    axes[1].scatter([clusteranzahl], [current.Inertia], color='red', s=90)
    axes[1].set_xlabel('k')
    axes[1].set_ylabel('Inertia')
    axes[1].set_title('Elbow: niedriger ist besser, Knick suchen')

    axes[2].plot(metrics_df['k'], metrics_df['Silhouette'], marker='o')
    axes[2].scatter([best_k], [metrics_df['Silhouette'].max()], color='green', s=110, label='bestes k')
    axes[2].scatter([clusteranzahl], [current.Silhouette], color='red', s=90, label='aktuelles k')
    axes[2].set_xlabel('k')
    axes[2].set_ylabel('Silhouette')
    axes[2].set_title('Silhouette: höher ist besser')
    axes[2].legend()
    plt.tight_layout()
    plt.show()

widgets.interact(
    clustering_demo,
    clusteranzahl=widgets.IntSlider(value=3, min=2, max=8),
    cluster_overlap=widgets.FloatSlider(value=0.20, min=0.02, max=0.80, step=0.02),
    seed=widgets.IntSlider(value=42, min=1, max=99),
);
